# Generating Scenario 1, 2 and 3

This script shows how I generate the hypothetical Scenario 1, 2 and 3
- **Scenario 1**: Reduced In-Vehicle Travel Times for moderate and high frequency patterns by 15%  

- **Scenario 2**: Hypothetical trips injected at the midpoint of the headway between each two consecutive trips that serve moderate frequency patterns, with travel time of that hypothetical trip set at the median travel time for trips serving that pattern for that day

- **Scenario 3**: Scenario 1 followed by Scenario 2 together

<br>
Each scenario has their 'A' and 'B' versions:

- **'A' Version**: Scenario applied to the eligible patterns serving the entire study area

- **'B' Version**: Scenario applied to the eligible patterns that are EITHER city centre-bound OR serving most deprived MSOAs

<br>
Specifically for Manchester, the 'A' version is already limited to patterns operated by Bee Network, to reflect the already-franchised nature of buses there (see scenario_utils.py for more info).  


---

For this notebook, there are TWO main parts:

1) Preparing Scenario Registries

2) Scenario Generation

In [ ]:
from pathlib import Path
import os

from scenario_utils import (
    make_pattern_filter, load_morning_patterns, load_baseline_gtfs,
    build_scenario_output, write_gtfs_zip, summarize_scenario_run,
)

ROOT = Path("../Data")
ROOT.resolve()

post_processed = ROOT/'Postprocessed'
baseline_gtfs = ROOT/'Scenario 0_Baseline'

reduced_gtfs = ROOT/'Scenario 1_Reduced Journeys'
frequency_gtfs = ROOT/'Scenario 2_Increased Frequencies'
compounded_gtfs = ROOT/'Scenario 3_Compounded'

for output_dir in [reduced_gtfs, frequency_gtfs, compounded_gtfs]:
    os.makedirs(output_dir, exist_ok=True)

target_cities = ['Bristol', 'Leeds', 'Manchester']

target_dates = [
    '20260603', '20260610',
    '20260617', '20260624'
]

## Part 1: Preparing Scenario Registries

Each entry names its output folder/label and the `speedup_filters` / `inject_filters` dict(s)
to use.  

A dict of `{None: filter_fn}` means "whole study area, no suffix in the output filename" (the "a" scenarios)  
A dict of `{'inbound': ..., 'deprived': ...}` means "selected areas only, one output file per suffix" (the "b" scenarios).

- `frequent?` values eligible for the **speed-up** stage: `frequent` + `not_frequent`
- `frequent?` values eligible for **injection**: `not_frequent` only

In [ ]:
SPEEDUP_ALL = make_pattern_filter(['frequent', 'not_frequent'])
INJECT_ALL = make_pattern_filter(['not_frequent'])

SPEEDUP_AREA = {
    area: make_pattern_filter(['frequent', 'not_frequent'], area=area)
    for area in ['inbound', 'deprived']
}
INJECT_AREA = {
    area: make_pattern_filter(['not_frequent'], area=area)
    for area in ['inbound', 'deprived']
}

SCENARIOS = [
    {   # 1a - Reduced Journey Times for entire study area
        'key': '1a', 'label': 'reduced', 'output_dir': reduced_gtfs,
        'speedup_filters': {None: SPEEDUP_ALL}, 'inject_filters': None,
    },
    {   # 1b - Reduced Journey Times for selected areas
        'key': '1b', 'label': 'reduced', 'output_dir': reduced_gtfs,
        'speedup_filters': SPEEDUP_AREA, 'inject_filters': None,
    },
    {   # 2a - Increased Frequencies for entire study area
        'key': '2a', 'label': 'frequent', 'output_dir': frequency_gtfs,
        'speedup_filters': None, 'inject_filters': {None: INJECT_ALL},
    },
    {   # 2b - Increased Frequencies for selected areas
        'key': '2b', 'label': 'frequent', 'output_dir': frequency_gtfs,
        'speedup_filters': None, 'inject_filters': INJECT_AREA,
    },
    {   # 3a - Compounded (1a then 2a) for entire study area
        'key': '3a', 'label': 'compounded', 'output_dir': compounded_gtfs,
        'speedup_filters': {None: SPEEDUP_ALL}, 'inject_filters': {None: INJECT_ALL},
    },
    {   # 3b - Compounded (1b then 2b) for selected areas
        'key': '3b', 'label': 'compounded', 'output_dir': compounded_gtfs,
        'speedup_filters': SPEEDUP_AREA, 'inject_filters': INJECT_AREA,
    },
]

## Part 2: Scenario Generation

In [ ]:
for scenario in SCENARIOS:

    suffixes = list((scenario['speedup_filters'] or scenario['inject_filters']).keys())

    for city in target_cities:

        patterns = load_morning_patterns(post_processed, city)

        for target_date in target_dates:

            stop_times, trips, retro_input = load_baseline_gtfs(baseline_gtfs, city, target_date)

            for suffix in suffixes:

                speedup_fn = scenario['speedup_filters'][suffix] if scenario['speedup_filters'] else None
                inject_fn = scenario['inject_filters'][suffix] if scenario['inject_filters'] else None

                new_trips, new_stoptimes, n_modified, n_injected, n_patterns = build_scenario_output(
                    patterns, stop_times, trips,
                    speedup_filter=speedup_fn, inject_filter=inject_fn
                )

                out_name = (
                    f"{city}_{target_date}_"
                    + (f"{suffix}_" if suffix else "")
                    + f"{scenario['label']}.zip"
                )
                write_gtfs_zip(scenario['output_dir']/out_name, retro_input, new_trips, new_stoptimes)

                print(summarize_scenario_run(
                    city, target_date, suffix,
                    speedup_active=speedup_fn is not None,
                    inject_active=inject_fn is not None,
                    trips_modified=n_modified,
                    trips_injected=n_injected,
                    patterns_injected=n_patterns,
                ))